# Llama Document-Aware Extraction Demo

## YAML-First Architecture with Document-Specific Field Filtering

This notebook demonstrates the refactored `llama_document_aware.py` system with **document-aware field filtering** that intelligently selects relevant fields based on document type.

### 📋 Complete Semantically Grouped Field Schema (49 fields)
The system supports extraction of these fields from `evaluation_data/ground_truth.csv`:

**Document Identification:**
`DOCUMENT_TYPE`, `INVOICE_NUMBER`, `RECEIPT_NUMBER`

**Business Details:**
`SUPPLIER_NAME`, `BUSINESS_ABN`, `BUSINESS_ADDRESS`, `BUSINESS_PHONE`, `SUPPLIER_WEBSITE`, `SUPPLIER_EMAIL`

**Customer Information:**
`PAYER_NAME`, `PAYER_ABN`, `PAYER_ADDRESS`, `PAYER_PHONE`, `PAYER_EMAIL`

**Temporal Data:**
`INVOICE_DATE`, `DUE_DATE`, `TRANSACTION_DATE`, `STATEMENT_DATE_RANGE`, `CREDIT_CARD_DUE_DATE`

**Line Items:**
`LINE_ITEM_DESCRIPTIONS`, `LINE_ITEM_QUANTITIES`, `LINE_ITEM_PRICES`, `LINE_ITEM_TOTAL_PRICES`, `LINE_ITEM_GST_AMOUNTS`, `LINE_ITEM_DISCOUNT_AMOUNTS`

**Financial Totals:**
`SUBTOTAL_AMOUNT`, `TOTAL_DISCOUNT_AMOUNT`, `GST_AMOUNT`, `IS_GST_INCLUDED`, `TOTAL_AMOUNT`, `TOTAL_AMOUNT_PAID`, `BALANCE_OF_PAYMENT`, `TOTAL_AMOUNT_PAYABLE`

**Payment & Location:**
`PAYMENT_METHOD`, `STORE_LOCATION`

**Banking Information:**
`BANK_NAME`, `BANK_BSB_NUMBER`, `BANK_ACCOUNT_NUMBER`, `BANK_ACCOUNT_HOLDER`, `ACCOUNT_OPENING_BALANCE`, `ACCOUNT_CLOSING_BALANCE`

**Transactions:**
`TOTAL_CREDITS`, `TOTAL_DEBITS`, `NET_TRANSACTION_AMOUNT`, `TRANSACTION_DATES`, `TRANSACTION_AMOUNTS_PAID`, `TRANSACTION_AMOUNTS_RECEIVED`, `TRANSACTION_BALANCES`

### 🎯 Document-Aware Field Filtering

**Key Innovation:** Instead of processing all 49 fields universally, the system intelligently filters to document-specific subsets:

- **📄 Invoice Documents**: 29 relevant fields (41% reduction)
- **🧾 Receipt Documents**: 20 relevant fields (59% reduction)  
- **🏦 Bank Statements**: 16 relevant fields (67% reduction)

This targeted approach delivers:
- ⚡ **Faster processing** with fewer fields to extract
- 🎯 **Higher accuracy** with document-specific prompts
- 💡 **Better resource utilization** focusing on relevant data
- 📈 **Improved performance** through YAML-first prompt configuration

---

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
from IPython.display import Image, display, HTML
import time

# Set project root (notebook is now in project root)
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Import our document-aware system
from llama_document_aware import DocumentAwareLlamaHandler
from common.evaluation_metrics import load_ground_truth

print("📚 Libraries imported successfully")
print(f"🗂️ Project root: {project_root}")
print(f"📍 Current directory: {Path.cwd()}")

## Configuration

Set up the document-aware processor with your model path:

In [ ]:
# Configuration - UPDATE WITH YOUR MODEL PATH
# MODEL_PATH = "/efs/shared/PTM/Llama-3.2-11B-Vision-Instruct"
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# TEST_IMAGE = "evaluation_data/image_041.png"  # INVOICE example
# TEST_IMAGE = "evaluation_data/image_004.png"  # RECEIPT example
TEST_IMAGE = "evaluation_data/image_006.png"  # Bank statement example

GROUND_TRUTH_PATH = "evaluation_data/ground_truth.csv"

# Initialize the document-aware handler
print("🚀 Initializing Document-Aware Llama Handler...")
handler = DocumentAwareLlamaHandler(MODEL_PATH, debug=True)
print("✅ Handler initialized successfully")

## Demo Image

Let's examine the test document we'll be processing:

In [ ]:
# Display the test image
image_path = project_root / TEST_IMAGE
if image_path.exists():
    display(Image(str(image_path), width=800))
    print(f"📄 Document: {image_path.name}")
else:
    print(f"❌ Image not found: {image_path}")
    print("Please update the TEST_IMAGE path above")

## Step 1: Document Type Detection

The system first detects the document type to select appropriate field schema:

In [ ]:
# Step 1: Detect document type and get schema
print("🔍 STEP 1: Document Type Detection")
print("=" * 50)

classification_info = handler.detect_and_classify_document(str(image_path))

print(f"\n📋 Document Type: {classification_info['document_type']}")
print(f"📊 Field Count: {classification_info['field_count']} fields (vs 49 universal)")
print(f"⚡ Efficiency: {(49 - classification_info['field_count'])/49*100:.0f}% fewer fields than universal approach")
print(f"\n🎯 Target Fields: {', '.join(classification_info['field_names'][:5])}...")

## Step 2: YAML-First Prompt Generation

The system generates a high-performance prompt using YAML configuration:

In [ ]:
# Step 2: Show the YAML-first prompt generation
print("📝 STEP 2: YAML-First Prompt Generation")
print("=" * 50)

# Access the processor to show the prompt (for demo purposes)
if handler.processor:
    # Reconfigure processor for the detected fields
    handler.processor.field_list = classification_info['field_names']
    handler.processor.field_count = len(classification_info['field_names'])
    
    # Generate the prompt
    prompt = handler.processor.generate_dynamic_prompt()
    
    print(f"🎯 Generated prompt for {len(classification_info['field_names'])} fields")
    print(f"📏 Prompt length: {len(prompt)} characters")
    print("\n🔍 YAML-FIRST PROMPT PREVIEW:")
    print("-" * 80)
    # Show first 500 characters of prompt
    print(prompt[:500] + "...\n[truncated for display]")
    print("-" * 80)
else:
    print("⚠️ Processor not initialized yet")

## Step 3: Document-Aware Extraction

Process the document with type-specific field extraction:

In [ ]:
# Step 3: Extract with document-aware processing
print("🔍 STEP 3: Document-Aware Extraction")
print("=" * 50)

start_time = time.time()
result = handler.process_document_aware(str(image_path), classification_info)
processing_time = time.time() - start_time

print(f"\n⏱️ Processing Time: {result['processing_time']:.3f}s")
print(f"📊 Fields Found: {result['detected_fields']}/{result['total_fields']}")
print(f"📈 Field Coverage: {result['detected_fields']/result['total_fields']*100:.1f}%")

## Step 4: Extracted Data Visualization

Display the extracted data with visual status indicators:

In [ ]:
# Step 4: Display extracted data with visual indicators
print("📊 STEP 4: Extracted Data Results")
print("=" * 80)

extracted_data = result["extracted_data"]
found_count = 0
total_count = len(extracted_data)

for field_name, value in extracted_data.items():
    if value != "NOT_FOUND":
        status = "✅"
        found_count += 1
    else:
        status = "❌"
    
    print(f"{status} {field_name}: {value}")

print("\n" + "=" * 80)
print(f"📈 EXTRACTION SUMMARY:")
print(f"   ✅ Fields Found: {found_count}/{total_count}")
print(f"   📊 Success Rate: {found_count/total_count*100:.1f}%")
print(f"   ⏱️ Processing Time: {result['processing_time']:.3f}s")
print(f"   🎯 Document Type: {result['document_type']}")
print(f"   ⚡ Field Reduction: {(49-total_count)/49*100:.0f}% fewer fields than universal")

## Step 5: Ground Truth Evaluation

Compare against ground truth data for accuracy measurement:

In [ ]:
# Step 5: Evaluate against ground truth
print("📊 STEP 5: Ground Truth Evaluation")
print("=" * 50)

# Load ground truth
ground_truth_path = project_root / GROUND_TRUTH_PATH
if ground_truth_path.exists():
    ground_truth = load_ground_truth(str(ground_truth_path))
    image_name = Path(image_path).name
    
    if image_name in ground_truth:
        # Evaluate with document-type-specific metrics
        evaluation_report = handler.evaluate_document_aware([result], ground_truth)
        
        # Extract metrics for this document type
        doc_type = result['document_type']
        type_metrics = evaluation_report['by_document_type'].get(doc_type, {})
        
        print(f"\n📈 ACCURACY RESULTS:")
        print(f"   🎯 Overall Accuracy: {type_metrics.get('avg_accuracy', 0)*100:.1f}%")
        print(f"   ✅ Meets Threshold: {'Yes' if type_metrics.get('meets_threshold', 0) > 0 else 'No'}")
        print(f"   🔥 Critical Fields Perfect: {'Yes' if type_metrics.get('critical_perfect', 0) > 0 else 'No'}")
        
        if doc_type == 'invoice':
            ato_compliant = type_metrics.get('ato_compliant', 0) > 0
            print(f"   🏛️ ATO Compliant: {'Yes' if ato_compliant else 'No'}")
        
        print(f"\n📊 PERFORMANCE COMPARISON:")
        print(f"   🆚 Document-Aware: {type_metrics.get('avg_accuracy', 0)*100:.1f}% accuracy")
        print(f"   📉 Field Reduction: {(49-total_count)/49*100:.0f}% fewer fields")
        print(f"   ⚡ Faster Processing: Type-specific extraction")
        
    else:
        print(f"⚠️ No ground truth available for {image_name}")
else:
    print(f"❌ Ground truth file not found: {ground_truth_path}")